# 第 1 天练习 —— Brandon 网站摘要示例

## 练习目标（理念）

把 **Day 1** 的两条能力串起来：

1. 用 OpenAI Chat Completions 发一条简单消息（先打通 API）
2. 用自写爬虫 `fetch_website_contents` 抓网页正文，再让模型做摘要并 `display(Markdown(...))`

目标站点示例：`https://notyetfitjair.blog`（可改成你关心的 URL）。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `OPENAI_API_KEY` + `.env` | `load_dotenv` + 密钥形态检查 |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages`（system / user） | `messages_for(website)` 拼装角色 |
| 网页抓取 → 摘要 | `fetch_website_contents` → `summarize` → `display_summary` |

## 怎么跑

1. 准备好同目录的 `scraper.py` 与 `.env`（含 `OPENAI_API_KEY`）
2. 从上到下依次运行单元格（Shift+Enter）
3. 若改了 `scraper.py`，用后面注释里的 `importlib.reload` 热重载，不必重启内核
4. 想换网站：改 `summarize(...)` / `display_summary(...)` 里的 URL 即可


In [ ]:
# ========== 导入：后面网页抓取 + 调 OpenAI 都靠这些 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 importlib：改完 scraper.py 后可 reload，无需重启 Jupyter 内核
import importlib
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 导入抓取函数：用 HTTP/解析拿到网站正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里把模型输出渲染成 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 若本格报错：先去同文件夹的 troubleshooting notebook 排查环境/依赖
# 【注】If you get an error running this cell, then please head over to the troubleshooting notebook!

# 提示：改完 scraper.py 后，可取消下面注释做热重载（reload），不必 Restart Kernel
# TIP: If you modify scraper.py, run this to reload without restarting kernel:
# import scraper
# importlib.reload(scraper)
# from scraper import fetch_website_contents


In [ ]:
# ========== 环境：加载 .env 并做 API Key 形态自检 ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量取出 OpenAI 密钥（名字必须是 OPENAI_API_KEY，和官方 SDK 默认一致）
api_key = os.getenv('OPENAI_API_KEY')

# ========== 检查 API Key：缺了 / 前缀不对 / 首尾空白，分别给出提示 ==========

# 完全没读到密钥
if not api_key:
    # 错误提示字符串保持英文（与课程 troubleshooting 文案一致，勿改译以免对不上文档）
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 读到了但不像项目密钥（常见应以 sk-proj- 开头）
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 首尾有空格/制表符：复制粘贴进 .env 时的常见坑
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 通过上述启发式检查（仍不保证密钥一定有效，只是形态看起来正常）
    print("API key looks good so far!")


In [ ]:
# ========== 预览：先发一条最简单的 user 消息，确认 API 通 ==========

# 预览：用下面 messages 调 OpenAI 就这么简单；若有问题去 Troubleshooting notebook
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

# 发给模型的用户文本（保留英文：影响模型回答的内容不翻译）
message = "Hello, GPT! This is my first ever message to you! Hi, I am brandon"

# Chat Completions 的 messages：这里只有一条 role=user
messages = [{"role": "user", "content": message}]

# 在笔记本里直接显示 messages，方便确认结构（list[dict]）
messages


In [ ]:
# ========== 第一次真正调用：创建客户端并用当前 messages 要一次回复 ==========

# 创建 OpenAI 客户端；默认从环境变量 OPENAI_API_KEY 读密钥
openai = OpenAI()

# 调用 Chat Completions：model 用 gpt-5-nano；messages 用上一格准备好的列表
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 取出第一条 choice 里 assistant 的文本内容（.content）
response.choices[0].message.content


In [ ]:
# ========== 试爬虫工具：抓目标博客正文，先 print 看原始文本 ==========

# 试一下抓取工具 fetch_website_contents（同目录 scraper.py）
# Let's try out this utility

# 传入 URL，返回网站正文（字符串）；URL 保持原样，改站点只改这里
web = fetch_website_contents("https://notyetfitjair.blog")
# 打印抓到的原文，检查是否为空 / 是否含导航噪声
print(web)


In [ ]:
# ========== 网站摘要流水线：prompt → messages → summarize → Markdown 展示 ==========

# 步骤 1：准备 system / user 提示词（发给模型的英文 prompt 必须保留，改译会改变行为）
# Create your prompts

# 再创建一个客户端（本格可独立重跑；密钥仍来自环境变量）
openai = OpenAI()  # Initialize the OpenAI client

# system_prompt：规定「只做摘要、用 bullet、输出干净 Markdown、不要包代码块」
system_prompt = """
Your job is to analyze the content of a website, and only give me a summarized version of the content. Please use bullet points and make it very understandable. Clean the output and do not wrap the markdown in a code block.
"""
# user_prompt：前缀说明；真正网页正文会在 messages_for 里拼到后面
user_prompt = """
Here are the contents of a website.
Provide a summary of this website and make it look like a blog post with an h1 title.

"""

# 步骤 2：先做一个最小 messages 列表做一次试调用（注意：这里还没用上面的 system_prompt）
# Make the messages list

messages = [
    # system：定助手身份（帮助总结博客）
    {"role": "system", "content": "You are a helpful assistant that helps summarize blogs"},
    # user：占位问题（作者原先的试探调用，不是网站摘要本身）
    {"role": "user", "content": "What is 2 + 20?"}
]

# 用 gpt-4.1-nano 试跑上面那条算术消息，确认第二条调用通路
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 打印试探回复；后面才进入真正的网站摘要函数
response.choices[0].message.content # fill this in

def messages_for(website):
    """把 system_prompt +（user_prompt 与网站正文）拼成标准 messages 列表。"""
    return [
        # system：摘要规则（上面定义的长英文 prompt）
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 爬虫抓到的 website 正文
        {"role": "user", "content": user_prompt + website}
    ]

def summarize(url):
    """给定 URL：抓取 → 组 messages → Chat Completions → 返回摘要字符串。"""
    # 先抓网页正文
    website = fetch_website_contents(url)
    # 再让模型按 messages_for 的规则生成摘要
    response = openai.chat.completions.create(
        model = "gpt-4.1-nano",
        messages = messages_for(website)
    )
    # 返回 assistant 文本
    return response.choices[0].message.content

# 直接调用一次：对示例博客做摘要并作为本格表达式结果
summarize("https://notyetfitjair.blog")

def display_summary(url):
    """summarize 后用 Markdown 在笔记本里漂亮展示。"""
    # 拿到摘要字符串
    summary = summarize(url)
    # 渲染为 Markdown（比纯 print 更适合带标题/列表的输出）
    display(Markdown(summary))

# 端到端展示：抓取 → 摘要 → 笔记本里显示
display_summary("https://notyetfitjair.blog")
